In [36]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader
import torch
from transformers import AutoTokenizer, AutoModel
import torch.nn as nn
from torch.optim import AdamW
from torch.nn import BCEWithLogitsLoss
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, precision_score, recall_score, confusion_matrix, classification_report, roc_curve, auc
from peft import LoraConfig, get_peft_model  # For efficient fine-tuning



In [61]:
df = pd.read_excel('bioactivity_dataset_cleaned_outliers.xlsx')
df['mol'] = df['canonical_smiles'].apply(Chem.MolFromSmiles)
df = df.dropna(subset=['mol'])

# Compute/scale descriptors (if not already)
def compute_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return np.zeros(4)
    return np.array([Descriptors.MolWt(mol), Descriptors.MolLogP(mol),
                     rdMolDescriptors.CalcNumHBD(mol), rdMolDescriptors.CalcNumHBA(mol)])

df[['MW', 'LogP', 'NumHDonors', 'NumHAcceptors']] = df['canonical_smiles'].apply(compute_descriptors).tolist()

scaler = StandardScaler()
desc_cols = ['MW', 'LogP', 'NumHDonors', 'NumHAcceptors']
df[desc_cols] = scaler.fit_transform(df[desc_cols])

In [62]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5667 entries, 0 to 5666
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   canonical_smiles   5667 non-null   object 
 1   MW                 5667 non-null   float64
 2   LogP               5667 non-null   float64
 3   NumHDonors         5667 non-null   float64
 4   NumHAcceptors      5667 non-null   float64
 5   pIC50              5667 non-null   float64
 6   bioactivity_class  5667 non-null   object 
 7   bioactivity        5667 non-null   int64  
 8   mol                5667 non-null   object 
dtypes: float64(5), int64(1), object(3)
memory usage: 398.6+ KB


In [63]:
df.head(3)

,canonical_smiles,MW,LogP,NumHDonors,NumHAcceptors,pIC50,bioactivity_class,bioactivity,mol
0,O=C(CCCCCC(NC(=O)OCc1ccccc1)C(=O)Nc1cccc2cccnc...,0.524109,0.241522,1.32487,0.282913,8.6,active,1,<rdkit.Chem.rdchem.Mol object at 0x00000283838...
1,O=C(CCCCCC(C(=O)Nc1ccc2ncccc2c1)C(=O)Nc1ccc2nc...,0.732961,0.621072,1.32487,0.282913,9.0,active,1,<rdkit.Chem.rdchem.Mol object at 0x00000283F56...
2,O=C(/C=C/c1cccc(C(C(=O)Nc2ccccc2)C(=O)Nc2ccccc...,0.036571,-0.025903,1.32487,-0.713216,9.0,active,1,<rdkit.Chem.rdchem.Mol object at 0x00000283F56...


In [64]:
# # Augmentation (3 variants for ~15k samples)
# def augment_smiles(smiles, num_aug=3):
#     mol = Chem.MolFromSmiles(smiles)
#     if mol is None: return [smiles]
#     variants = [Chem.MolToSmiles(mol, canonical=True)]
#     for _ in range(num_aug):
#         try:
#             variant = Chem.MolToSmiles(mol, canonical=False, doRandom=True)
#             if Chem.MolFromSmiles(variant) is not None:
#                 variants.append(variant)
#         except: pass
#     return variants

# aug_df = []
# for _, row in df.iterrows():
#     for var_smiles in augment_smiles(row['canonical_smiles']):
#         aug_row = row.copy()
#         aug_row['canonical_smiles'] = var_smiles
#         aug_df.append(aug_row)
# df = pd.DataFrame(aug_df)

In [67]:
df.drop(columns=['pIC50', 'bioactivity_class'], inplace=True)

In [68]:
df.head(2)

,canonical_smiles,MW,LogP,NumHDonors,NumHAcceptors,bioactivity,mol
0,O=C(CCCCCC(NC(=O)OCc1ccccc1)C(=O)Nc1cccc2cccnc...,0.524109,0.241522,1.32487,0.282913,1,<rdkit.Chem.rdchem.Mol object at 0x00000283838...
1,O=C(CCCCCC(C(=O)Nc1ccc2ncccc2c1)C(=O)Nc1ccc2nc...,0.732961,0.621072,1.32487,0.282913,1,<rdkit.Chem.rdchem.Mol object at 0x00000283F56...


In [70]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5667 entries, 0 to 5666
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   canonical_smiles  5667 non-null   object 
 1   MW                5667 non-null   float64
 2   LogP              5667 non-null   float64
 3   NumHDonors        5667 non-null   float64
 4   NumHAcceptors     5667 non-null   float64
 5   bioactivity       5667 non-null   int64  
 6   mol               5667 non-null   object 
dtypes: float64(4), int64(1), object(2)
memory usage: 310.0+ KB


In [51]:
(df['canonical_smiles'])

0       O=C(CCCCCC(NC(=O)OCc1ccccc1)C(=O)Nc1cccc2cccnc...
0       n1c2c(NC(=O)C(CCCCCC(=O)NO)NC(=O)OCc3ccccc3)cc...
0       n1c2c(ccc1)cccc2NC(C(NC(=O)OCc1ccccc1)CCCCCC(=...
0       c1c(NC(C(NC(=O)OCc2ccccc2)CCCCCC(NO)=O)=O)c2nc...
1       O=C(CCCCCC(C(=O)Nc1ccc2ncccc2c1)C(=O)Nc1ccc2nc...
                              ...                        
5665      C(NO)(=O)/C=C/c1cc(Nc2ncnc3c2cc2c(c3)OCCO2)ccc1
5666             CN1CCc2c(c3ccccc3n2Cc2ccc(C(=O)NO)cc2)C1
5666           c1(ccc(Cn2c3CCN(C)Cc3c3c2cccc3)cc1)C(NO)=O
5666           C(c1ccc(cc1)Cn1c2ccccc2c2CN(C)CCc21)(=O)NO
5666             c1cccc2c1c1CN(CCc1n2Cc1ccc(cc1)C(=O)NO)C
Name: canonical_smiles, Length: 22627, dtype: object

In [71]:
train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df['bioactivity'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)
tokenizer = AutoTokenizer.from_pretrained('DeepChem/ChemBERTa-77M-MTR')

class BioactivityDataset(Dataset):
    def __init__(self, df, tokenizer, desc_scaler):
        self.smiles = df['canonical_smiles'].tolist()
        self.labels = df['bioactivity'].tolist()
        self.descs = df[desc_cols].values
        self.desc_scaler = desc_scaler
        self.encodings = tokenizer(self.smiles, truncation=True, padding=True, max_length=512, return_tensors='pt')
    
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        item['descriptors'] = torch.tensor(self.descs[idx], dtype=torch.float)
        return item

In [72]:
train_dataset = BioactivityDataset(train_df, tokenizer, scaler)
val_dataset = BioactivityDataset(val_df, tokenizer, scaler)
test_dataset = BioactivityDataset(test_df, tokenizer, scaler)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)
test_loader = DataLoader(test_dataset, batch_size=16)

In [73]:
train_dataset[0]

{'input_ids': tensor([12, 16, 23, 16, 17, 22, 19, 18, 39, 16, 17, 16, 16, 16, 16, 16, 16, 16,
         17, 22, 19, 18, 23, 15, 20, 15, 15, 15, 15, 17, 31, 15, 21, 15, 15, 15,
         15, 15, 21, 18, 15, 20, 18, 22, 23, 60, 19, 13,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0]),
 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
   

In [74]:
train_df['bioactivity'].value_counts(), val_df['bioactivity'].value_counts(), test_df['bioactivity'].value_counts()

(bioactivity
 1    3712
 0     821
 Name: count, dtype: int64,
 bioactivity
 1    454
 0    113
 Name: count, dtype: int64,
 bioactivity
 1    475
 0     92
 Name: count, dtype: int64)

In [56]:
class FineTunedMultimodalModel(nn.Module):
    def __init__(self, num_desc=4, emb_dim=384, hidden_dim=256, dropout=0.4, num_layers_to_unfreeze=2):
        super().__init__()
        self.smiles_encoder = AutoModel.from_pretrained('DeepChem/ChemBERTa-77M-MTR')

        self.config = self.smiles_encoder.config
        self.config.num_labels = 1  # Optional: Set for clas    sification task
        
        # Freeze all except last num_layers_to_unfreeze
        for param in self.smiles_encoder.parameters():
            param.requires_grad = False
        unfreeze_modules = self.smiles_encoder.encoder.layer[-num_layers_to_unfreeze:]
        for module in unfreeze_modules:
            for param in module.parameters():
                param.requires_grad = True
        
        self.desc_proj = nn.Linear(num_desc, emb_dim)
        self.fusion = nn.Sequential(
            nn.Linear(emb_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, input_ids=None, attention_mask=None, descriptors=None, **kwargs):
        outputs = self.smiles_encoder(input_ids=input_ids, attention_mask=attention_mask, **kwargs)
        smiles_emb = outputs.last_hidden_state[:, 0, :]
        desc_emb = self.desc_proj(descriptors)
        fused = torch.cat([smiles_emb, desc_emb], dim=-1)
        return self.fusion(fused).squeeze(-1)

model = FineTunedMultimodalModel(num_layers_to_unfreeze=1)  # Start with 2 for small data
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

Some weights of RobertaModel were not initialized from the model checkpoint at DeepChem/ChemBERTa-77M-MTR and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


FineTunedMultimodalModel(
  (smiles_encoder): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(600, 384, padding_idx=1)
      (position_embeddings): Embedding(515, 384, padding_idx=1)
      (token_type_embeddings): Embedding(1, 384)
      (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.144, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-2): 3 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=384, out_features=384, bias=True)
              (key): Linear(in_features=384, out_features=384, bias=True)
              (value): Linear(in_features=384, out_features=384, bias=True)
              (dropout): Dropout(p=0.109, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=384, out_features=384, bias=True)
          

In [57]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig( task_type=TaskType.SEQ_CLS, r=4, lora_alpha=8, target_modules=["query", "value"], lora_dropout=0.1)

In [21]:
model = get_peft_model(model, lora_config)
 
# Print trainable params (should be <1% of total)
model.print_trainable_parameters()  

trainable params: 18,432 || all params: 3,677,681 || trainable%: 0.5012


In [58]:
from sklearn.utils.class_weight import compute_class_weight
class_weights = compute_class_weight('balanced', classes=np.unique(train_df['bioactivity']), y=train_df['bioactivity'])
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)  # Low LR for fine-tuning
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)

criterion = BCEWithLogitsLoss(pos_weight=class_weights[1])
epochs = 50  # Longer for fine-tuning
best_auc = 0
patience, counter = 5, 0

In [59]:
for epoch in range(epochs):
    model.train()
    train_loss = 0
    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        descriptors = batch['descriptors'].to(device)
        labels = batch['labels'].float().to(device)
        
        optimizer.zero_grad()
        logits = model(input_ids=input_ids, attention_mask=attention_mask, descriptors=descriptors)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    scheduler.step(train_loss / len(train_loader))  # Adjust LR
    
    # Validation
    model.eval()
    val_preds, val_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            descriptors = batch['descriptors'].to(device)
            labels = batch['labels'].to(device)
            logits = model(input_ids, attention_mask, descriptors)
            probs = torch.sigmoid(logits).cpu().numpy()
            val_preds.extend(probs)
            val_labels.extend(labels.cpu().numpy())
    
    auc = roc_auc_score(val_labels, val_preds)
    print(f'Epoch {epoch+1}: Train Loss {train_loss/len(train_loader):.4f}, Val AUROC {auc:.4f}')
    
    if auc > best_auc:
        best_auc = auc
        torch.save(model.state_dict(), 'best_finetuned_model_.pth')
        counter = 0
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping!")
            break

Epoch 1: Train Loss 0.2955, Val AUROC 0.8855
Epoch 2: Train Loss 0.2224, Val AUROC 0.9211
Epoch 3: Train Loss 0.1853, Val AUROC 0.9419
Epoch 4: Train Loss 0.1589, Val AUROC 0.9561
Epoch 5: Train Loss 0.1444, Val AUROC 0.9621
Epoch 6: Train Loss 0.1258, Val AUROC 0.9680
Epoch 7: Train Loss 0.1163, Val AUROC 0.9713
Epoch 8: Train Loss 0.1060, Val AUROC 0.9750


KeyboardInterrupt: 

In [ ]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score

# Run inference on test set
model.eval()
test_preds = []
test_labels = []
with torch.no_grad():
	for batch in test_loader:
		input_ids = batch['input_ids'].to(device)
		attention_mask = batch['attention_mask'].to(device)
		descriptors = batch['descriptors'].to(device)
		labels = batch['labels'].to(device)
		logits = model(input_ids, attention_mask, descriptors)
		probs = torch.sigmoid(logits).cpu().numpy()
		test_preds.extend((probs > 0.5).astype(int))
		test_labels.extend(labels.cpu().numpy())

accuracy = balanced_accuracy_score(test_labels, test_preds)
print(f"Accuracy: {accuracy}")
print("accuracy another: ", accuracy_score(test_labels, test_preds))

Accuracy: 0.8932151029748284
accuracy another:  0.9312169312169312


In [33]:
print(classification_report(test_labels, test_preds))

              precision    recall  f1-score   support

           0       0.76      0.84      0.80        92
           1       0.97      0.95      0.96       475

    accuracy                           0.93       567
   macro avg       0.87      0.89      0.88       567
weighted avg       0.93      0.93      0.93       567



In [34]:
roc_auc_score(test_labels, test_preds)

0.8932151029748283

In [21]:
# Recreate the model architecture
model_1 = FineTunedMultimodalModel(num_layers_to_unfreeze=1)
model_1.load_state_dict(torch.load('best_finetuned_model.pth', map_location='cpu'))  # or 'cuda' if using GPU
model_1.to(device)
model_1.eval()

Some weights of RobertaModel were not initialized from the model checkpoint at DeepChem/ChemBERTa-77M-MTR and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


FineTunedMultimodalModel(
  (smiles_encoder): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(600, 384, padding_idx=1)
      (position_embeddings): Embedding(515, 384, padding_idx=1)
      (token_type_embeddings): Embedding(1, 384)
      (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.144, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-2): 3 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=384, out_features=384, bias=True)
              (key): Linear(in_features=384, out_features=384, bias=True)
              (value): Linear(in_features=384, out_features=384, bias=True)
              (dropout): Dropout(p=0.109, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=384, out_features=384, bias=True)
          

In [22]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score

# Run inference on test set

test_preds = []
test_labels = []
with torch.no_grad():
	for batch in test_loader:
		input_ids = batch['input_ids'].to(device)
		attention_mask = batch['attention_mask'].to(device)
		descriptors = batch['descriptors'].to(device)
		labels = batch['labels'].to(device)
		logits = model_1(input_ids, attention_mask, descriptors)
		probs = torch.sigmoid(logits).cpu().numpy()
		test_preds.extend((probs > 0.5).astype(int))
		test_labels.extend(labels.cpu().numpy())

accuracy = balanced_accuracy_score(test_labels, test_preds)
print(f"Accuracy: {accuracy}")
print("accuracy another: ", accuracy_score(test_labels, test_preds))
roc_auc = roc_auc_score(test_labels, test_preds)
print(f"ROC AUC: {roc_auc}")

Accuracy: 0.9560995049920296
accuracy another:  0.9730921923246582
ROC AUC: 0.9560995049920296
